# pandas 02. 選ぶ・絞る・変える

`[]` `.loc` `.iloc` の違い、代入の落とし穴、`map`/`apply` の使い分け。

セルを実行する前に、どうなるか予想する。

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

NULLISH = {"NULL", "N/A", "-", ""}

def load():
    df = pd.read_csv("/data/orders.csv", dtype=str, keep_default_na=False)
    df = df.map(lambda s: None if str(s).strip() in NULLISH else s)
    df["qty"] = pd.to_numeric(df["qty"]).astype("Int64")
    df["amount"] = pd.to_numeric(df["amount"].str.replace(",", "", regex=False)).astype("Int64")
    return df

df = load()
df

---
## 1. 列を選ぶ

### 1-1. df['col'] と df[['col']]

角括弧が1つか2つかで、返ってくるものの型が変わる。

**実行する前に、違いを予想する。**

In [ ]:
# A: 1つ
a = df["amount"]
print(type(a).__name__)
a.head(3)

In [ ]:
# B: 2つ
b = df[["amount"]]
print(type(b).__name__)
b.head(3)

<details>
<summary>何が起きたか</summary>

- `A` は **Series**(1次元)。`.sum()` や `.str` が使える
- `B` は **DataFrame**(2次元)。列が1つだけの表

リストを渡せば複数列を選べて、**その順に並ぶ**。

```python
df[["amount", "order_id"]]      # この順に並び替わる
```

出力の列順を仕様に合わせるときは、この書き方でまとめて指定するのが早い。

使い分け:
- 1列を計算に使う → `A`
- 表として扱う、列を絞る → `B`

</details>

### 1-2. .loc と .iloc

ラベルで取るか、位置で取るか。スライスの端の扱いも違う。

**実行する前に、違いを予想する。**

In [ ]:
# A: .loc (ラベル)
print(df.loc[0:2, "order_id"])
print("---")
print(df.loc[df["region"] == "east", ["order_id", "amount"]])

In [ ]:
# B: .iloc (位置)
print(df.iloc[0:2, 0])
print("---")
print(df.iloc[[0, 2], [0, 6]])

<details>
<summary>何が起きたか</summary>

**`.loc` のスライスは右端を含む。** `0:2` は3行(0,1,2)。
`.iloc` は Python の普通のスライスなので `0:2` は2行(0,1)。

| | 指定するもの | スライスの右端 |
| --- | --- | --- |
| `.loc` | ラベル(index の値、列名) | **含む** |
| `.iloc` | 位置(0始まりの整数) | 含まない |

いまは index が 0,1,2... なので `.loc` のラベルもたまたま整数だが、
**index を振り直したり並べ替えたりすると一致しなくなる**。

`.loc` は条件式も受け取れる。`df.loc[条件, 列]` が一番よく使う形。

</details>

---
## 2. 行を絞る

### 2-1. and は使えない

複数条件をつなぐとき、Python の `and` ではなく `&` を使う。括弧も要る。

**実行する前に、違いを予想する。**

In [ ]:
# A: and (落ちる)
try:
    df[(df["region"] == "east") and (df["status"] == "completed")]
except Exception as e:
    print(f"{type(e).__name__}: {e}")

In [ ]:
# B: & と括弧
df[(df["region"] == "east") & (df["status"] == "completed")]

<details>
<summary>何が起きたか</summary>

`and` は左右を**単一の真偽値**に変換しようとする。Series は
「全部Trueか」「1つでもTrueか」が曖昧なので `ValueError` になる。

`&` `|` `~` は**要素ごと**に働く。

括弧が要るのは演算子の優先順位のため。`&` は `==` より強いので、
`df["region"] == "east" & df["status"] == "completed"` は
`df["region"] == ("east" & df["status"]) == "completed"` と解釈されて壊れる。

**条件は必ず括弧で囲む。** 例外なく。

</details>

### 2-2. isin / between / query

同じ絞り込みの書き方いろいろ。

**実行する前に、違いを予想する。**

In [ ]:
# A: & でつなぐ
a = df[(df["region"] == "east") | (df["region"] == "west")]
display(a)
b = df[(df["amount"] >= 1000) & (df["amount"] <= 3000)]
b

In [ ]:
# B: isin / between / query
a = df[df["region"].isin(["east", "west"])]
display(a)
b = df[df["amount"].between(1000, 3000)]
display(b)
c = df.query("region in ['east','west'] and amount >= 1000")
c

<details>
<summary>何が起きたか</summary>

結果は同じ。`isin` と `between` は条件が増えるほど読みやすくなる。

- `between(a, b)` は**両端を含む**(`inclusive="both"` が既定)。
  片側だけにしたいなら `inclusive="left"` など
- `query` は文字列で書く。中では `and` / `or` が使える(`&` も可)。
  変数を参照するときは `@` を付ける: `df.query("amount > @limit")`

`query` は短く書けるが、文字列なのでエディタの補完も型チェックも効かない。
**短い条件は `query`、複雑なものは `&`** くらいの使い分けでよい。

</details>

### 2-3. 欠損を含む列の比較

条件式は欠損をどう扱うか。SQL とは少し違う。

**実行する前に、違いを予想する。**

In [ ]:
# A: 比較する
print(df["amount"].isna().sum(), "件が欠損")
a = df[df["amount"] > 1000]
print(len(a), "行")
a[["order_id", "amount"]]

In [ ]:
# B: 否定してみる
b = df[~(df["amount"] > 1000)]
print(len(b), "行")
b[["order_id", "amount"]]

<details>
<summary>何が起きたか</summary>

**欠損の行は A にも B にも入らない。** 足しても全行にならない。

`NaN > 1000` は `False` になる(SQLの三値論理と違い、pandas の比較は
欠損に対して `False` を返す)。だから `~` で反転しても拾われない。

欠損も残したいなら明示する。

```python
df[(df["amount"] > 1000) | df["amount"].isna()]
```

**「条件の否定」は「欠損を含む補集合」ではない。** これは行数が合わない事故の常連。

</details>

---
## 3. 代入の落とし穴

### 3-1. 連鎖indexing

絞った結果に代入する、をどう書くか。片方は効かないことがある。

**実行する前に、違いを予想する。**

In [ ]:
# A: 連鎖して代入
a = load()
sub = a[a["region"] == "east"]
sub["region"] = "EAST"       # 警告が出るかもしれない
print("元のdfは変わったか:")
print(a[a["region"].isin(["east", "EAST"])]["region"].tolist())

In [ ]:
# B: .loc で一度に指定
b = load()
b.loc[b["region"] == "east", "region"] = "EAST"
print("元のdfは変わったか:")
print(b["region"].tolist())

<details>
<summary>何が起きたか</summary>

`A` の `sub` は元の `a` の**コピーかもしれないし、ビュー(参照)かもしれない**。
どちらになるかは pandas の内部事情で決まるので、**元が変わるかどうかが予測できない**。
pandas はこれを `SettingWithCopyWarning` で警告する。

`B` は「どの行の、どの列に」を1回の `.loc` で指定しているので曖昧さがない。

規則はこれだけ:

- **元を変えたい** → `df.loc[条件, 列] = 値`
- **別物を作りたい** → `sub = df[条件].copy()` と `.copy()` を明示する

`df[条件][列] = 値` のように**角括弧を2回続けて代入しない**。

</details>

### 3-2. assign は元を変えない

列を足す2つの書き方。

**実行する前に、違いを予想する。**

In [ ]:
# A: 直接代入
a = load()
a["total"] = a["qty"] * a["amount"]
print(a.columns.tolist())

In [ ]:
# B: assign
b = load()
c = b.assign(total=b["qty"] * b["amount"])
print("元:", b.columns.tolist())
print("新:", c.columns.tolist())

<details>
<summary>何が起きたか</summary>

`A` は元の `a` を書き換える。`B` の `assign` は**新しい DataFrame を返す**。

`assign` はメソッドチェーンの途中に挟めるのが利点。

```python
(df
 .assign(total=lambda d: d["qty"] * d["amount"])
 .query("total > 3000")
 .sort_values("total"))
```

`lambda d:` を使うと**その時点の DataFrame** を参照できる。
直前の `assign` で作った列もつなげて使える。

短いスクリプトなら `A` で十分。**変換が5段以上続くなら `B`** のほうが追いやすい。

</details>

---
## 4. 値を変換する

### 4-1. map / apply / str アクセサ

1列の全要素に処理をかける3つの書き方。欠損の扱いが違う。

**実行する前に、違いを予想する。**

In [ ]:
# A: map と apply
s = df["region"]
print("map      :", s.map(lambda x: x.upper() if x else x).tolist())
print("apply    :", s.apply(lambda x: x.upper() if x else x).tolist())
print("---- 欠損を含む列で ----")
t = df["customer_id"]
print("map(str.upper) :")
try:
    print(t.map(str.upper).tolist())
except Exception as e:
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# B: .str アクセサ
t = df["customer_id"]
print(t.str.upper().tolist())
print("---")
print(df["region"].str.lower().tolist())

<details>
<summary>何が起きたか</summary>

- `Series.map` と `Series.apply` は、1列に対してはほぼ同じ。
  `map` は辞書も受け取れる(`s.map({"east": "東"})`)
- **`.str.*` は欠損を素通りさせる。** `None` はそのまま `None` で返る

`map(str.upper)` は欠損に対して `str.upper(None)` を呼ぼうとして落ちる。
`map` には `na_action="ignore"` があり、これを付けると欠損を飛ばす。

```python
t.map(str.upper, na_action="ignore")
```

**文字列操作は `.str` を第一候補にする。** 欠損に強く、速く、短い。
`.str` に無いことをやるときだけ `map` を使う。

</details>

### 4-2. apply の axis

DataFrame に対する `apply` は、行に効くか列に効くか。

**実行する前に、違いを予想する。**

In [ ]:
# A: axis=0 (既定)
a = df[["qty", "amount"]].apply(lambda col: col.max())
print(type(a).__name__)
a

In [ ]:
# B: axis=1
b = df[["qty", "amount"]].apply(lambda row: row["qty"] * row["amount"], axis=1)
print(type(b).__name__)
b.head()

<details>
<summary>何が起きたか</summary>

- `axis=0`(既定)は**列ごと**に関数を呼ぶ。渡ってくるのは1列ぶんの Series
- `axis=1` は**行ごと**。渡ってくるのは1行ぶんの Series

`axis` の意味は「どの軸に沿って潰すか」。0が行方向(=列ごとに集計)。
覚えにくいので、**迷ったら小さいデータで両方試す**。

なお `axis=1` の `apply` は**1行ずつ Python の関数を呼ぶので非常に遅い**。
`df["qty"] * df["amount"]` と書けるならそちらを使う。
数万行を超えると体感で差が出る。

</details>

### 4-3. where / mask / np.where

条件で値を差し替える3つ。どちらを残すかが逆になる。

**実行する前に、違いを予想する。**

In [ ]:
# A: where と mask
s = df["amount"]
print("where:", s.where(s > 1000, 0).tolist())
print("mask :", s.mask(s > 1000, 0).tolist())

In [ ]:
# B: np.where
s = df["amount"]
try:
    print(np.where(s > 1000, "高", "低"))
except Exception as e:
    print(f"{type(e).__name__}: {e}")

# 欠損を先に埋めれば通る
print(np.where(s.fillna(0) > 1000, "高", "低"))

<details>
<summary>何が起きたか</summary>

- `s.where(cond, other)` は **cond が True の値を残す**。False を `other` に
- `s.mask(cond, other)` は逆。**cond が True の値を差し替える**
- `np.where(cond, a, b)` は三項演算子。True なら `a`、False なら `b`

`where` の引数の順番は直感に反しやすい。**「where=残す、mask=隠す」**と覚える。

**`np.where` は nullable な型(`Int64` など)を扱えない。**
`s > 1000` の結果に `pd.NA` が混ざると、NumPy がその真偽を決められずに落ちる。
`fillna` で潰すか、`where`/`mask` を使う。pandas 側の関数は `pd.NA` を理解する。

`np.where` は NumPy の配列を返すので、index も dtype も失う。
`pd.Series(np.where(...), index=s.index)` と包み直すか、
3つ以上に分岐するなら `np.select` を使う。

欠損は `cond` が `False` 扱いになるので、`where` では `other` に置き換わる。
残したいなら `s.where(cond | s.isna(), other)`。

</details>

### 4-4. replace と str.replace

名前は似ているが、別物。

**実行する前に、違いを予想する。**

In [ ]:
# A: replace (値まるごと)
s = pd.Series(["east", "west", "eastern"])
print(s.replace("east", "E").tolist())
print(s.replace({"east": "E", "west": "W"}).tolist())

In [ ]:
# B: str.replace (部分文字列)
s = pd.Series(["east", "west", "eastern"])
print(s.str.replace("east", "E", regex=False).tolist())
print(s.str.replace(r"^e", "E", regex=True).tolist())

<details>
<summary>何が起きたか</summary>

- `Series.replace` は**値そのもの**が一致したときだけ置換する。
  `"eastern"` は変わらない。辞書で対応表を渡せる
- `Series.str.replace` は**部分文字列**を置換する。`"eastern"` → `"Eern"` になる

`str.replace` の `regex` は pandas 2.x で既定が `False`。
正規表現を使いたいなら**明示的に `regex=True`**。
昔のコードは既定が `True` だったので、コピペすると挙動が変わる。

コード体系の正規化(`E`→`east`)は `replace` の辞書、
表記の掃除(カンマや円記号を落とす)は `str.replace` の正規表現、と使い分ける。

</details>

---
## 練習

In [ ]:
# 練習1: status が completed で、amount が 1000 以上の行を、
#        order_id と amount の2列だけにして取り出す。

ans = ...   # ここに書く

assert list(ans.columns) == ["order_id", "amount"], f"列が違う: {list(ans.columns)}"
assert len(ans) == 5, f"5行のはず: {len(ans)}"
assert ans["amount"].sum() == 18000
print("OK")

In [ ]:
# 練習2: region を小文字に正規化した列 region_norm を、
#        元の df を書き換えずに追加した新しい DataFrame を作る。
#        (欠損があっても落ちないこと)

ans = ...   # ここに書く

assert "region_norm" not in df.columns, "元の df を書き換えている"
assert sorted(ans["region_norm"].unique()) == ["east", "north", "south", "west"]
print("OK")

In [ ]:
# 練習3: amount が 2000 以上なら "high"、未満なら "low"、
#        欠損なら "unknown" になる Series を作る。

ans = ...   # ここに書く

assert ans.tolist() == ["high", "low", "high", "high", "low", "unknown",
                        "high", "low", "high", "low", "low", "low"], ans.tolist()
print("OK")

---
## まとめ

| 書き方 | 意味 | 注意 |
| --- | --- | --- |
| `df["c"]` | Series | 1次元 |
| `df[["c"]]` | DataFrame | リストの順に並ぶ |
| `.loc[行, 列]` | ラベルで指定 | **スライスの右端を含む** |
| `.iloc[行, 列]` | 位置で指定 | 右端を含まない |
| `&` `|` `~` | 要素ごとの論理演算 | **括弧が要る**。`and` は使えない |
| `isin` / `between` | 読みやすい絞り込み | `between` は両端を含む |
| `df.loc[cond, col] = v` | 安全な代入 | 角括弧を2回続けない |
| `.copy()` | 明示的にコピー | 別物として扱いたいとき |
| `assign` | 列を足した新しい表を返す | チェーンの途中に置ける |
| `.str.*` | 文字列操作 | **欠損を素通りする**。第一候補 |
| `map(na_action="ignore")` | 欠損を飛ばす | `.str` で足りないとき |
| `apply(axis=1)` | 行ごと | **遅い**。ベクトル化できないか先に考える |
| `where` / `mask` | 残す / 隠す | 引数の意味が逆 |
| `replace` / `str.replace` | 値ごと / 部分文字列 | `regex=` を明示する |

次: `pandas-03-group-and-dedup.ipynb`